# Load RAG Eval Results
Auto-discovers all `testingevals*` / `timingevals*` directory pairs under `BASE_OUT` (from `config.py`) and loads them into DataFrames. A `gpu_suffix` column tracks which run each row came from.

In [28]:
import os
import re
import json
import pandas as pd
from pathlib import Path
from config import BASE_OUT

BASE = Path(BASE_OUT)

# Auto-discover all testingevals* dirs and pair with their timingevals* counterpart
run_dirs = []
for d in sorted(BASE.glob("testingevals*")):
    suffix = d.name[len("testingevals"):]
    timing = BASE / f"timingevals{suffix}"
    if timing.exists():
        run_dirs.append((d, timing, suffix))

print(f"Found {len(run_dirs)} run(s):")
for td, timd, suf in run_dirs:
    print(f"  suffix={suf!r}  testing={td.name}  timing={timd.name}")

Found 1 run(s):
  suffix='_local'  testing=testingevals_local  timing=timingevals_local


In [29]:
pd.set_option('display.max_columns', None)


In [30]:
# Filename format (fields separated by ~):
# {model}~{dataset}~rag{use_rag}~k{topk}~comp{compress}~{compress_model}~r{compress_r}
# ~rerank{rerank}~n{rerank_k}~{rerank_model}~ret{index_model}~{text_data}~{index_data}
# ~batch{batch}~{batch_size}~pipe{pipeline}~dev{device_config}

CONFIG_KEYS = [
    "model", "dataset",
    "use_rag", "topk",
    "compress", "compress_model", "compress_r",
    "rerank", "rerank_k", "rerank_model",
    "index_model", "text_data", "index_data",
    "batch", "batch_size", "pipeline", "device_config",
]

def parse_config(stem: str) -> dict:
    """Parse RAG config from a filename stem (no prefix_, no extension)."""
    parts = stem.split("~")
    if len(parts) != len(CONFIG_KEYS):
        raise ValueError(f"Expected {len(CONFIG_KEYS)} fields, got {len(parts)}: {stem}")
    cfg = dict(zip(CONFIG_KEYS, parts))
    # Use removeprefix to strip exact leading tokens (not char-by-char like lstrip)
    cfg["use_rag"]       = int(cfg["use_rag"].removeprefix("rag"))
    cfg["topk"]          = int(cfg["topk"].removeprefix("k"))
    cfg["compress"]      = int(cfg["compress"].removeprefix("comp"))
    cfg["compress_r"]    = float(cfg["compress_r"].removeprefix("r"))
    cfg["rerank"]        = int(cfg["rerank"].removeprefix("rerank"))
    cfg["rerank_k"]      = int(cfg["rerank_k"].removeprefix("n"))
    cfg["index_model"]   = cfg["index_model"].removeprefix("ret")
    cfg["batch"]         = int(cfg["batch"].removeprefix("batch"))
    cfg["batch_size"]    = int(cfg["batch_size"])
    cfg["pipeline"]      = cfg["pipeline"].removeprefix("pipe")
    cfg["device_config"] = cfg["device_config"].removeprefix("dev")
    return cfg

def stem_from_path(path: Path, prefix: str) -> str:
    """Strip file prefix and extension to get the config stem."""
    name = path.stem  # e.g. results_Llama-3.2-1B~...
    return name[len(prefix) + 1:]  # +1 for the underscore

# Quick sanity check
sample = next(TESTING_DIR.glob("results_*.csv"))
print(parse_config(stem_from_path(sample, "results")))

{'model': 'Llama-3.2-3B-Instruct', 'dataset': 'squad_dataset', 'use_rag': 1, 'topk': 1, 'compress': 0, 'compress_model': 'none', 'compress_r': 1.0, 'rerank': 0, 'rerank_k': 0, 'rerank_model': 'none', 'index_model': 'e5-base-v2', 'text_data': 'test_sample_2018_sent', 'index_data': 'e5_Flat_index', 'batch': 1, 'batch_size': 1, 'pipeline': 'standard', 'device_config': '187789'}


## 1. Testing Results (`testingevals{OUT_SUFFIX}/results_*.csv`)
Columns: `gold, prediction, rouge, f1, em, recall` + config columns

In [31]:
result_frames = []
for testing_dir, _, gpu_suffix in run_dirs:
    for f in sorted(testing_dir.glob("results_*.csv")):
        cfg = parse_config(stem_from_path(f, "results"))
        df = pd.read_csv(f)
        df.insert(0, "idx", range(len(df)))
        for k, v in cfg.items():
            df[k] = v
        df["gpu_suffix"] = gpu_suffix
        result_frames.append(df)

df_results = pd.concat(result_frames, ignore_index=True)
print(f"df_results: {df_results.shape}")
df_results.head()

df_results: (9000, 25)


,idx,gold,prediction,rouge,f1,em,recall,model,dataset,use_rag,topk,compress,compress_model,compress_r,rerank,rerank_k,rerank_model,index_model,text_data,index_data,batch,batch_size,pipeline,device_config,gpu_suffix
0,0,"['Dennis Hull, as well as painter Manley MacDo...",- Michael Caine,0.0,0.0,0,NaN,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,0,none,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,_local
1,1,['Narcissa'],Tryphosa,0.0,0.0,0,NaN,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,0,none,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,_local
2,2,['yes'],"Karjiang is a mountain, and Sherpi Kangri is a...",0.0,0.0,0,NaN,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,0,none,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,_local
3,3,['about 7 million'],"1,000",0.0,0.0,0,NaN,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,0,none,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,_local
4,4,['neo-Nazi'],Dysania,0.0,0.0,0,NaN,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,0,none,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,_local


## 2. Timing / Performance (`timingevals{OUT_SUFFIX}/perfiii_*.csv`)
Each row = one query run; many timing/GPU columns + config columns.

In [32]:
import ast

def expand_dict_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Expand dict-valued columns (string or real dicts) into flat columns, recursively."""
    while True:
        to_drop, new_cols = [], {}
        for col in df.columns:
            sample = df[col].dropna()
            if sample.empty:
                continue
            v = sample.iloc[0]
            # Handle both string-serialized dicts and actual dict objects
            if isinstance(v, str) and v.strip().startswith('{'):
                try:
                    parsed = df[col].map(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
                except Exception:
                    continue
            elif isinstance(v, dict):
                parsed = df[col]
            else:
                continue
            try:
                keys = list(parsed.dropna().iloc[0].keys())
                for k in keys:
                    suffix = f"gpu{k}" if isinstance(k, int) else k
                    new_cols[f"{col}_{suffix}"] = parsed.map(
                        lambda d, k=k: d.get(k) if isinstance(d, dict) else None
                    )
                to_drop.append(col)
            except Exception:
                pass
        if not to_drop:
            break
        df = df.drop(columns=to_drop)
        df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
    return df


perf_frames = []
for _, timing_dir, gpu_suffix in run_dirs:
    for f in sorted(timing_dir.glob("perfiii_*.csv")):
        cfg = parse_config(stem_from_path(f, "perfiii"))
        df = pd.read_csv(f)
        df = expand_dict_cols(df)
        for k, v in cfg.items():
            df[k] = v
        df["gpu_suffix"] = gpu_suffix
        perf_frames.append(df)

df_perf = pd.concat(perf_frames, ignore_index=True)
print(f"df_perf: {df_perf.shape}")
df_perf.head()

df_perf: (9000, 134)


,rag_total_timer,rag_total,ttft,decode_time,generate_total_time,generate,tokens_generated,tokens_per_second,rag_total_timer_iter_0,rag_total_iter_0,ttft_iter_0,decode_time_iter_0,generate_total_time_iter_0,generate_iter_0,tokens_generated_iter_0,tokens_per_second_iter_0,question,prediction,mode,idx,cuda_visible_devices,num_gpus,gpu_0_name,init_cpu_start,load_embedder_cpu_before,load_embedder_cpu_after,embedder_device,load_generator_cpu_before,load_generator_cpu_after,generator_device,rerank,rerank_model,rerank_top_n,faiss_nvml_after_gpu_0_used_mb,faiss_nvml_after_gpu_0_free_mb,gen_gpu_before_allocated,gen_gpu_before_peak,gen_gpu_all_before_gpu_0_allocated,gen_gpu_all_before_gpu_0_peak,gen_gpu_all_after_gpu_0_allocated,gen_gpu_all_after_gpu_0_peak,gen_gpu_after_allocated,gen_gpu_after_peak,faiss_nvml_after_gpu_0_iter_0_used_mb,faiss_nvml_after_gpu_0_iter_0_free_mb,gen_gpu_before_iter_0_allocated,gen_gpu_before_iter_0_peak,gen_gpu_all_before_gpu_0_iter_0_allocated,gen_gpu_all_before_gpu_0_iter_0_peak,gen_gpu_all_after_gpu_0_iter_0_allocated,gen_gpu_all_after_gpu_0_iter_0_peak,gen_gpu_after_iter_0_allocated,gen_gpu_after_iter_0_peak,energy_j_gpu0,power_mean_w_gpu0,init_gpu_start_gpu_0_allocated,init_gpu_start_gpu_0_peak,load_embedder_gpu_after_allocated,load_embedder_gpu_after_peak,load_generator_gpu_after_allocated,load_generator_gpu_after_peak,util_mean_gpu0_sm_pct,util_mean_gpu0_mem_pct,util_peak_gpu0_sm_pct,util_peak_gpu0_mem_pct,model,dataset,use_rag,topk,compress,compress_model,compress_r,rerank_k,index_model,text_data,index_data,batch,batch_size,pipeline,device_config,gpu_suffix,embed_timer,embed,faiss_timer,faiss,context_tokens_before,context_tokens_after,embed_timer_iter_0,embed_iter_0,faiss_timer_iter_0,faiss_iter_0,context_tokens_before_iter_0,context_tokens_after_iter_0,load_corpus_cpu_before,load_corpus_cpu_after,load_faiss_cpu_before,load_faiss_cpu_after,faiss_is_gpu,faiss_device,embed_gpu_before_allocated,embed_gpu_before_peak,embed_gpu_after_allocated,embed_gpu_after_peak,faiss_nvml_before_gpu_0_used_mb,faiss_nvml_before_gpu_0_free_mb,embed_gpu_before_iter_0_allocated,embed_gpu_before_iter_0_peak,embed_gpu_after_iter_0_allocated,embed_gpu_after_iter_0_peak,faiss_nvml_before_gpu_0_iter_0_used_mb,faiss_nvml_before_gpu_0_iter_0_free_mb,load_corpus_gpu_after_gpu_0_allocated,load_corpus_gpu_after_gpu_0_peak,load_faiss_gpu_before_gpu_0_allocated,load_faiss_gpu_before_gpu_0_peak,load_faiss_nvml_before_gpu_0_used_mb,load_faiss_nvml_before_gpu_0_free_mb,load_faiss_gpu_after_gpu_0_allocated,load_faiss_gpu_after_gpu_0_peak,load_faiss_nvml_after_gpu_0_used_mb,load_faiss_nvml_after_gpu_0_free_mb,tokenize_timer,tokenize,input_tokens_total,generate_timer,decode_timer,decode,tokenize_timer_iter_0,tokenize_iter_0,input_tokens_total_iter_0,generate_timer_iter_0,decode_timer_iter_0,decode_iter_0,batch_start
0,1.782359e+09,0.352845,0.304549,0.034423,0.338972,0.338972,5,145.252251,1.782359e+09,0.352845,0.304549,0.034423,0.338972,0.338972,5,145.252251,"Who else besides ""The Golden Jet"" was born in ...",- Michael Caine,single,0.0,NaN,1,NVIDIA GeForce RTX 3090,771.371094,868.164062,888.832031,cuda:0,888.640625,1238.175781,cuda:0,0,none,3,3641.8125,20934.1875,2911.747584,2911.747584,2911.747584,2911.747584,2920.267264,2944.819712,2920.267264,2944.819712,3641.8125,20934.1875,2911.747584,2911.747584,2911.747584,2911.747584,2920.267264,2944.819712,2920.267264,2944.819712,39.23325,112.0950,0.0,0.0,439.069696,439.069696,2911.747584,2911.747584,1.714286,0.0,4,0,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,_local,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.782359e+09,0.065590,0.041187,0.022833,0.064020,0.064020,4,175.187862,1.782359e+09,0.065590,0.041187,0.022833,0.064020,0.064020,4,175.187862,"Wha

## 3. Traces (`timingevals{OUT_SUFFIX}/trace_*.jsonl`)
Each line = one query.  
- `meta`: question, mode, idx  
- `trace_stage_durations`: per-stage timing (e.g. `generate`)  
- `samples`: time-series GPU/CPU power — aggregated here to mean/peak per GPU

In [33]:
trace_rows = []

for _, timing_dir, gpu_suffix in run_dirs:
    for f in sorted(timing_dir.glob("trace_*.jsonl")):
        cfg = parse_config(stem_from_path(f, "trace"))
        with open(f) as fh:
            for line in fh:
                line = line.strip()
                if not line:
                    continue
                rec = json.loads(line)
                row = dict(cfg)
                row["gpu_suffix"] = gpu_suffix

                # Meta fields
                meta = rec.get("meta", {})
                row["question"] = meta.get("question")
                row["mode"]     = meta.get("mode")
                row["idx"]      = meta.get("idx")

                # Markers (e.g. generate_start, generate_end timestamps)
                for marker, t in (rec.get("markers") or {}).items():
                    row[f"marker_{marker}"] = t

                # Stage durations
                for stage, dur in (rec.get("trace_stage_durations") or {}).items():
                    row[f"dur_{stage}"] = dur

                # Aggregate all sample sensor columns: mean and peak
                samples = rec.get("samples") or []
                if samples:
                    sdf = pd.DataFrame(samples).drop(columns=["t"], errors="ignore")
                    for col in sdf.columns:
                        row[f"{col}_mean"] = sdf[col].mean()
                        row[f"{col}_peak"] = sdf[col].max()

                trace_rows.append(row)

df_trace = pd.DataFrame(trace_rows)
print(f"df_trace: {df_trace.shape}")
df_trace.head()

df_trace: (2100, 40)


,model,dataset,use_rag,topk,compress,compress_model,compress_r,rerank,rerank_k,rerank_model,index_model,text_data,index_data,batch,batch_size,pipeline,device_config,gpu_suffix,question,mode,idx,marker_generate_start,marker_generate_end,dur_generate,gpu0_power_w_mean,gpu0_power_w_peak,gpu0_sm_pct_mean,gpu0_sm_pct_peak,gpu0_mem_pct_mean,gpu0_mem_pct_peak,gpu0_mem_mb_mean,gpu0_mem_mb_peak,cpu_pct_mean,cpu_pct_peak,marker_embed_start,marker_embed_end,marker_faiss_start,marker_faiss_end,dur_embed,dur_faiss
0,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,0,none,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,_local,"Who else besides ""The Golden Jet"" was born in ...",single,0.0,0.000796,0.352579,0.351783,112.984343,115.821,2.149254,4,0.000000,0,3714.051306,3737.8125,95.531343,389.3,NaN,NaN,NaN,NaN,NaN,NaN
1,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,0,none,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,_local,"What is the first name of the woman who, like ...",single,1.0,0.000453,0.065381,0.064928,117.086462,117.215,7.692308,8,1.846154,2,3756.274038,3757.8125,164.469231,1172.3,NaN,NaN,NaN,NaN,NaN,NaN
2,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,0,none,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,_local,Is Karjiang a mountain and Sherpi Kangri a mou...,single,2.0,0.000417,0.199924,0.199508,123.266949,128.891,13.512821,51,5.461538,29,3757.812500,3757.8125,106.435897,267.9,NaN,NaN,NaN,NaN,NaN,NaN
3,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,0,none,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,_local,How many visitors per year does the first stop...,single,3.0,0.000504,0.050979,0.050474,128.891000,128.891,51.000000,51,29.000000,29,3757.812500,3757.8125,95.020000,194.4,NaN,NaN,NaN,NaN,NaN,NaN
4,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,0,none,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,_local,What kind of ideology is Torment's drummer inv...,single,4.0,0.000386,0.048057,0.047672,135.935800,137.697,51.000000,51,29.000000,29,3757.812500,3757.8125,95.570000,195.1,NaN,NaN,NaN,NaN,NaN,NaN


## Summary

In [34]:
print("df_results columns:", list(df_results.columns))
print()
print("df_perf columns:",    list(df_perf.columns))
print()
print("df_trace columns:",   list(df_trace.columns))

df_results columns: ['idx', 'gold', 'prediction', 'rouge', 'f1', 'em', 'recall', 'model', 'dataset', 'use_rag', 'topk', 'compress', 'compress_model', 'compress_r', 'rerank', 'rerank_k', 'rerank_model', 'index_model', 'text_data', 'index_data', 'batch', 'batch_size', 'pipeline', 'device_config', 'gpu_suffix']

df_perf columns: ['rag_total_timer', 'rag_total', 'ttft', 'decode_time', 'generate_total_time', 'generate', 'tokens_generated', 'tokens_per_second', 'rag_total_timer_iter_0', 'rag_total_iter_0', 'ttft_iter_0', 'decode_time_iter_0', 'generate_total_time_iter_0', 'generate_iter_0', 'tokens_generated_iter_0', 'tokens_per_second_iter_0', 'question', 'prediction', 'mode', 'idx', 'cuda_visible_devices', 'num_gpus', 'gpu_0_name', 'init_cpu_start', 'load_embedder_cpu_before', 'load_embedder_cpu_after', 'embedder_device', 'load_generator_cpu_before', 'load_generator_cpu_after', 'generator_device', 'rerank', 'rerank_model', 'rerank_top_n', 'faiss_nvml_after_gpu_0_used_mb', 'faiss_nvml_after

In [35]:
# Sweep coverage summary for df_trace
# Key dimensions from testing_callsmain_sweep.py

KEY_DIMS = ["gpu_suffix", "model", "dataset", "use_rag", "topk", "batch", "pipeline"]

print("=== Unique values per sweep dimension ===")
for col in KEY_DIMS:
    if col in df_trace.columns:
        vals = sorted(df_trace[col].unique())
        print(f"  {col:15s}: {vals}")

print()
print("=== Rows per (gpu_suffix, model, dataset) ===")
coverage = (
    df_trace.groupby(["gpu_suffix", "model", "dataset"])
    .apply(lambda g: pd.Series({
        "rows":          len(g),
        "use_rag_vals":  sorted(g["use_rag"].unique()),
        "topk_vals":     sorted(g["topk"].unique()),
        "batch_vals":    sorted(g["batch"].unique()),
    }))
)
print(coverage.to_string())

print()
print("=== Missing (use_rag, topk) combos per group ===")
# Expected from sweep: no-RAG -> topk=0; RAG -> topk in [1,3,5,10]
expected = {(0, 0), (1, 1), (1, 3), (1, 5), (1, 10)}
for (gpu_suffix, model, dataset), g in df_trace.groupby(["gpu_suffix", "model", "dataset"]):
    found = set(zip(g["use_rag"].astype(int), g["topk"].astype(int)))
    missing = expected - found
    if missing:
        print(f"  MISSING {gpu_suffix} / {model} / {dataset}: {sorted(missing)}")
print("  (no output above = all expected combos present)")

=== Unique values per sweep dimension ===
  gpu_suffix     : ['_local']
  model          : ['Llama-3.2-1B-Instruct', 'Llama-3.2-3B-Instruct']
  dataset        : ['hotpotqa_dataset', 'nq_dataset', 'popqa_dataset', 'squad_dataset', 'triviaqa_dataset', 'webquestions_dataset']
  use_rag        : [0, 1]
  topk           : [0, 1, 3, 5, 10]
  batch          : [0, 1]
  pipeline       : ['standard']

=== Rows per (gpu_suffix, model, dataset) ===
                                                       rows use_rag_vals         topk_vals batch_vals
gpu_suffix model                 dataset                                                             
_local     Llama-3.2-1B-Instruct hotpotqa_dataset       150       [0, 1]  [0, 1, 3, 5, 10]        [0]
                                 nq_dataset             150       [0, 1]  [0, 1, 3, 5, 10]        [0]
                                 popqa_dataset          150       [0, 1]  [0, 1, 3, 5, 10]        [0]
                                 squad_dataset   

/tmp/ipykernel_63395/2318317887.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: pd.Series({


In [36]:
for name, df in [("df_results", df_results), ("df_perf", df_perf), ("df_trace", df_trace)]:
    with open(BASE / f"{name}_columns.txt", "w") as f:
        f.write("\n".join(df.columns))
    print(f"Wrote {name}_columns.txt ({len(df.columns)} columns)")

Wrote df_results_columns.txt (25 columns)
Wrote df_perf_columns.txt (134 columns)
Wrote df_trace_columns.txt (40 columns)


In [37]:
df_perf.head()

,rag_total_timer,rag_total,ttft,decode_time,generate_total_time,generate,tokens_generated,tokens_per_second,rag_total_timer_iter_0,rag_total_iter_0,ttft_iter_0,decode_time_iter_0,generate_total_time_iter_0,generate_iter_0,tokens_generated_iter_0,tokens_per_second_iter_0,question,prediction,mode,idx,cuda_visible_devices,num_gpus,gpu_0_name,init_cpu_start,load_embedder_cpu_before,load_embedder_cpu_after,embedder_device,load_generator_cpu_before,load_generator_cpu_after,generator_device,rerank,rerank_model,rerank_top_n,faiss_nvml_after_gpu_0_used_mb,faiss_nvml_after_gpu_0_free_mb,gen_gpu_before_allocated,gen_gpu_before_peak,gen_gpu_all_before_gpu_0_allocated,gen_gpu_all_before_gpu_0_peak,gen_gpu_all_after_gpu_0_allocated,gen_gpu_all_after_gpu_0_peak,gen_gpu_after_allocated,gen_gpu_after_peak,faiss_nvml_after_gpu_0_iter_0_used_mb,faiss_nvml_after_gpu_0_iter_0_free_mb,gen_gpu_before_iter_0_allocated,gen_gpu_before_iter_0_peak,gen_gpu_all_before_gpu_0_iter_0_allocated,gen_gpu_all_before_gpu_0_iter_0_peak,gen_gpu_all_after_gpu_0_iter_0_allocated,gen_gpu_all_after_gpu_0_iter_0_peak,gen_gpu_after_iter_0_allocated,gen_gpu_after_iter_0_peak,energy_j_gpu0,power_mean_w_gpu0,init_gpu_start_gpu_0_allocated,init_gpu_start_gpu_0_peak,load_embedder_gpu_after_allocated,load_embedder_gpu_after_peak,load_generator_gpu_after_allocated,load_generator_gpu_after_peak,util_mean_gpu0_sm_pct,util_mean_gpu0_mem_pct,util_peak_gpu0_sm_pct,util_peak_gpu0_mem_pct,model,dataset,use_rag,topk,compress,compress_model,compress_r,rerank_k,index_model,text_data,index_data,batch,batch_size,pipeline,device_config,gpu_suffix,embed_timer,embed,faiss_timer,faiss,context_tokens_before,context_tokens_after,embed_timer_iter_0,embed_iter_0,faiss_timer_iter_0,faiss_iter_0,context_tokens_before_iter_0,context_tokens_after_iter_0,load_corpus_cpu_before,load_corpus_cpu_after,load_faiss_cpu_before,load_faiss_cpu_after,faiss_is_gpu,faiss_device,embed_gpu_before_allocated,embed_gpu_before_peak,embed_gpu_after_allocated,embed_gpu_after_peak,faiss_nvml_before_gpu_0_used_mb,faiss_nvml_before_gpu_0_free_mb,embed_gpu_before_iter_0_allocated,embed_gpu_before_iter_0_peak,embed_gpu_after_iter_0_allocated,embed_gpu_after_iter_0_peak,faiss_nvml_before_gpu_0_iter_0_used_mb,faiss_nvml_before_gpu_0_iter_0_free_mb,load_corpus_gpu_after_gpu_0_allocated,load_corpus_gpu_after_gpu_0_peak,load_faiss_gpu_before_gpu_0_allocated,load_faiss_gpu_before_gpu_0_peak,load_faiss_nvml_before_gpu_0_used_mb,load_faiss_nvml_before_gpu_0_free_mb,load_faiss_gpu_after_gpu_0_allocated,load_faiss_gpu_after_gpu_0_peak,load_faiss_nvml_after_gpu_0_used_mb,load_faiss_nvml_after_gpu_0_free_mb,tokenize_timer,tokenize,input_tokens_total,generate_timer,decode_timer,decode,tokenize_timer_iter_0,tokenize_iter_0,input_tokens_total_iter_0,generate_timer_iter_0,decode_timer_iter_0,decode_iter_0,batch_start
0,1.782359e+09,0.352845,0.304549,0.034423,0.338972,0.338972,5,145.252251,1.782359e+09,0.352845,0.304549,0.034423,0.338972,0.338972,5,145.252251,"Who else besides ""The Golden Jet"" was born in ...",- Michael Caine,single,0.0,NaN,1,NVIDIA GeForce RTX 3090,771.371094,868.164062,888.832031,cuda:0,888.640625,1238.175781,cuda:0,0,none,3,3641.8125,20934.1875,2911.747584,2911.747584,2911.747584,2911.747584,2920.267264,2944.819712,2920.267264,2944.819712,3641.8125,20934.1875,2911.747584,2911.747584,2911.747584,2911.747584,2920.267264,2944.819712,2920.267264,2944.819712,39.23325,112.0950,0.0,0.0,439.069696,439.069696,2911.747584,2911.747584,1.714286,0.0,4,0,Llama-3.2-1B-Instruct,hotpotqa_dataset,0,0,0,none,1.0,0,e5-base-v2,test_sample_2018_sent,e5_Flat_index,0,1,standard,187789,_local,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.782359e+09,0.065590,0.041187,0.022833,0.064020,0.064020,4,175.187862,1.782359e+09,0.065590,0.041187,0.022833,0.064020,0.064020,4,175.187862,"Wha